In [1]:
import os
import gc
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, log_loss
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from scipy.stats import randint, uniform
from sklearn.decomposition import PCA


import xgboost as xgb
import lightgbm as lgb
import catboost as cb

#  Settings 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
N = 10   # ways
K = 3    # shots
Q = 4    # queries per class
EPISODES = 1000  # Few-shot episodes
PCA_DIM = 1280

In [2]:
#  Dataset 
class ProtoDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = np.array(file_paths)
        self.labels = np.array(labels)
        self.transform = transform
        self.classes = np.unique(labels)
        self.class_to_indices = {c: np.where(labels == c)[0] for c in self.classes}

    def sample_episode(self, N, K, Q):
        selected_classes = np.random.choice(self.classes, N, replace=False)
        support_images, support_labels, query_images, query_labels = [], [], [], []

        for i, cls in enumerate(selected_classes):
            idxs = np.random.choice(self.class_to_indices[cls], K + Q, replace=False)
            support_idxs, query_idxs = idxs[:K], idxs[K:]
            for idx in support_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                support_images.append(img)
                support_labels.append(i)
            for idx in query_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                query_images.append(img)
                query_labels.append(i)
        return (
            torch.stack(support_images),
            torch.tensor(support_labels),
            torch.stack(query_images),
            torch.tensor(query_labels)
        )

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

In [3]:
#  Load Dataset 
DATA_PATH = "C:/Users/MSC055/Desktop/Sabrina/CSE711/Eye Image Dataset"
CLASSES = sorted(os.listdir(DATA_PATH))

file_paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls)
    for f in os.listdir(cls_path):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            file_paths.append(os.path.join(cls_path, f))
            labels.append(idx)

file_paths = np.array(file_paths)
labels = np.array(labels)

X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

print(f"Train_size: {len(X_train)} | Val_size: {len(X_val)} | Test_size: {len(X_test)}")


Train_size: 3734 | Val_size: 800 | Test_size: 801


In [4]:
#  Prototypical Network 
class ProtoNetwork(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.gap = nn.AdaptiveAvgPool2d(1)


    def forward(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = self.gap(f)
            f = f.view(f.size(0), -1)
        return f

class ProtoLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, support, support_labels, query, query_labels, model):
        support_emb = model(support)
        query_emb = model(query)
        prototypes = []
        for cls in torch.unique(support_labels):
            prototypes.append(support_emb[support_labels==cls].mean(dim=0))
        prototypes = torch.stack(prototypes)
        dists = torch.cdist(query_emb, prototypes)
        logits = -dists
        return self.ce(logits, query_labels)

#  Feature Extraction Function 
def extract_embeddings(model, file_paths, transform, batch_size=16):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(file_paths), batch_size), desc="Extracting Features"):
            batch_paths = file_paths[i:i+batch_size]
            imgs = []
            for path in batch_paths:
                img = Image.open(path).convert("RGB")
                imgs.append(transform(img))
            imgs = torch.stack(imgs).to(DEVICE)
            feats = model(imgs).cpu().numpy().astype(np.float32)
            embeddings.append(feats)
            del imgs, feats
            gc.collect()
    return np.vstack(embeddings)

In [5]:
#  CNN Backbones for Feature Fusion 
backbones = {
    "resnet50": models.resnet50,
    "densenet121": models.densenet121,
    "mobilenetv2": models.mobilenet_v2,
    "efficientnet_b0": models.efficientnet_b0,
    "vgg16": models.vgg16
}

# Symbolic Models
symbolic_models = {
    "RandomForest": (
        RandomForestClassifier(random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(5, 50),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", None],
            "bootstrap": [True, False]
        }
    ),
    "XGBoost": (
        xgb.XGBClassifier(eval_metric="mlogloss", random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    
    "LightGBM": (
        lgb.LGBMClassifier(verbose=-1, random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    "CatBoost": (
        cb.CatBoostClassifier(verbose=0,random_state=SEED),
        {
            "iterations": randint(50, 100),
            "depth": randint(3, 12),
            "learning_rate": uniform(0.01, 0.3),
            "l2_leaf_reg": uniform(1, 10),
            "border_count": randint(32, 255),
            "random_strength": uniform(0, 10)
        }
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=SEED),
        {
           "n_estimators": randint(50, 100),
            "learning_rate": uniform(0.01, 1.0)
        }
    )
}


In [ ]:
#  Training + Feature Fusion 

# Prepare ProtoDataset
proto_dataset = ProtoDataset(X_train, y_train, transform=train_transform)

# Train Prototypical Networks for each backbone
proto_models = {}
for name, func in backbones.items():
    print(f"\nTraining backbone: {name}")
    model = func(pretrained=True)
    if hasattr(model, "fc"): model.fc = nn.Identity()
    if hasattr(model, "classifier"): model.classifier = nn.Identity()
    if hasattr(model, "head"): model.head = nn.Identity()
    model = nn.Sequential(*list(model.children())[:-1])
    model.to(DEVICE).eval()

    proto_model = ProtoNetwork(model).to(DEVICE)
    optimizer = torch.optim.Adam(proto_model.parameters(), lr=1e-4)
    proto_loss = ProtoLoss()

    print("Training Prototypical Network...")
    for episode in range(EPISODES):
        support_x, support_y, query_x, query_y = proto_dataset.sample_episode(N,K,Q)
        support_x, query_x = support_x.to(DEVICE), query_x.to(DEVICE)
        support_y, query_y = support_y.to(DEVICE), query_y.to(DEVICE)
        loss = proto_loss(support_x, support_y, query_x, query_y, proto_model)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if episode % 50 == 0:
            print(f"[{name}] Episode {episode}/{EPISODES} | Prototypical Loss: {loss.item():.4f}")

    proto_models[name] = proto_model
    torch.cuda.empty_cache()
    gc.collect()

#  Feature Fusion + PCA (TRAIN)
def fuse_features_train(models_dict, file_paths, transform, pca_dim=PCA_DIM):
    all_feats = []
    pca_models = {}

    for name, model in models_dict.items():
        print(f"\nExtracting TRAIN features for {name}...")
        feats = extract_embeddings(model, file_paths, transform)

        n_samples, feat_dim = feats.shape
        effective_pca_dim = min(pca_dim, feat_dim, n_samples)

        print(
            f"Applying PCA for {name} | "
            f"samples={n_samples}, feat_dim={feat_dim}, pca_dim={effective_pca_dim}"
        )

        pca = PCA(
            n_components=effective_pca_dim,
            whiten=True,
            random_state=SEED
        )

        feats_pca = pca.fit_transform(feats)

        all_feats.append(feats_pca)
        pca_models[name] = pca

        del feats, feats_pca
        gc.collect()

    fused = np.concatenate(all_feats, axis=1)
    print(f"\nFused TRAIN feature shape: {fused.shape}")

    return fused, pca_models

#  Feature Fusion + PCA (VAL)
def fuse_features_eval(models_dict, file_paths, transform, pca_models):
    all_feats = []

    for name, model in models_dict.items():
        print(f"\nExtracting VAL features for {name}...")
        feats = extract_embeddings(model, file_paths, transform)

        feats_pca = pca_models[name].transform(feats)
        all_feats.append(feats_pca)

        del feats, feats_pca
        gc.collect()

    fused = np.concatenate(all_feats, axis=1)
    print(f"\nFused VAL feature shape: {fused.shape}")

    return fused


# TRAIN FEATURES
X_train_feat, pca_models = fuse_features_train(
    proto_models, X_train, val_transform
)

# VALIDATION FEATURES
X_val_feat = fuse_features_eval(
    proto_models, X_val, val_transform, pca_models
)


#  Train Symbolic Models 
for sym_name, (clf, param_dist) in symbolic_models.items():
    print(f"\nTraining Symbolic Model: {sym_name}")
    search = RandomizedSearchCV(clf, param_distributions=param_dist, n_iter=10, cv=3,
                                scoring="accuracy", random_state=SEED, n_jobs=-1)
    search.fit(X_train_feat, y_train)
    tuned_model = search.best_estimator_
    best_params = search.best_params_

    y_train_pred = tuned_model.predict(X_train_feat)
    y_val_pred = tuned_model.predict(X_val_feat)
    y_train_proba = tuned_model.predict_proba(X_train_feat)
    y_val_proba = tuned_model.predict_proba(X_val_feat)

    train_loss_calc = log_loss(y_train, y_train_proba)
    val_loss_calc = log_loss(y_val, y_val_proba)
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    overall_acc = val_acc
    precision = precision_score(y_val, y_val_pred, average='macro', zero_division=0)
    recall = recall_score(y_val, y_val_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    class_report = classification_report(y_val, y_val_pred, zero_division=0)
    conf_matrix = confusion_matrix(y_val, y_val_pred)

    print(f"\nFeature Fusion + {sym_name} Results:")
    print(f"Train Loss: {train_loss_calc:.4f} | Val Loss: {val_loss_calc:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Accuracy: {overall_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"Best Params: {best_params}")
    print(f"Classification Report:\n{class_report}")
    print(f"Confusion Matrix:\n{conf_matrix}")


Training backbone: resnet50


c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Training Prototypical Network...
[resnet50] Episode 0/1000 | Prototypical Loss: 1.9699
[resnet50] Episode 50/1000 | Prototypical Loss: 0.9139
[resnet50] Episode 100/1000 | Prototypical Loss: 0.9925
[resnet50] Episode 150/1000 | Prototypical Loss: 0.6917
[resnet50] Episode 200/1000 | Prototypical Loss: 0.5951
[resnet50] Episode 250/1000 | Prototypical Loss: 0.5857
[resnet50] Episode 300/1000 | Prototypical Loss: 0.6588
[resnet50] Episode 350/1000 | Prototypical Loss: 0.4764
[resnet50] Episode 400/1000 | Prototypical Loss: 0.4617
[resnet50] Episode 450/1000 | Prototypical Loss: 0.2364
[resnet50] Episode 500/1000 | Prototypical Loss: 0.4712
[resnet50] Episode 550/1000 | Prototypical Loss: 0.2658
[resnet50] Episode 600/1000 | Prototypical Loss: 0.1843
[resnet50] Episode 650/1000 | Prototypical Loss: 0.4475
[resnet50] Episode 700/1000 | Prototypical Loss: 0.3043
[resnet50] Episode 750/1000 | Prototypical Loss: 0.4612
[resnet50] Episode 800/1000 | Prototypical Loss: 0.3130
[resnet50] Episode

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[densenet121] Episode 0/1000 | Prototypical Loss: 1.9252
[densenet121] Episode 50/1000 | Prototypical Loss: 0.9603
[densenet121] Episode 100/1000 | Prototypical Loss: 0.6845
[densenet121] Episode 150/1000 | Prototypical Loss: 0.4624
[densenet121] Episode 200/1000 | Prototypical Loss: 0.5749
[densenet121] Episode 250/1000 | Prototypical Loss: 0.7379
[densenet121] Episode 300/1000 | Prototypical Loss: 0.5230
[densenet121] Episode 350/1000 | Prototypical Loss: 0.6596
[densenet121] Episode 400/1000 | Prototypical Loss: 0.4359
[densenet121] Episode 450/1000 | Prototypical Loss: 0.4079
[densenet121] Episode 500/1000 | Prototypical Loss: 0.6188
[densenet121] Episode 550/1000 | Prototypical Loss: 0.2627
[densenet121] Episode 600/1000 | Prototypical Loss: 0.5169
[densenet121] Episode 650/1000 | Prototypical Loss: 0.5885
[densenet121] Episode 700/1000 | Prototypical Loss: 0.4946
[densenet121] Episode 750/1000 | Prototypical Loss: 0.3491
[densenet121] Episode 800/1000 | Prototypical Loss: 0.2331


c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


... (23 outputs left out)

Extracting Features: 100%|██████████| 50/50 [00:22<00:00,  2.23it/s]



Fused VAL feature shape: (800, 5376)

Training Symbolic Model: RandomForest

Feature Fusion + RandomForest Results:
Train Loss: 0.3263 | Val Loss: 0.8011 | Train Acc: 0.8822 | Val Acc: 0.7688
Accuracy: 0.7688 | Precision: 0.8161 | Recall: 0.6803 | F1: 0.7205
Best Params: {'bootstrap': True, 'max_depth': 6, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 13, 'n_estimators': 57}
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.40      0.55        15
           1       0.92      0.92      0.92       226
           2       0.87      0.68      0.76        19
           3       0.67      0.76      0.71       202
           4       0.77      0.69      0.73       153
           5       0.67      0.66      0.66        67
           6       0.62      0.64      0.63        75
           7       1.00      0.33      0.50         3
           8       0.86      0.95      0.90        19
           9       0.94      0.76    

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Feature Fusion + LightGBM Results:
Train Loss: 0.0085 | Val Loss: 1.0451 | Train Acc: 1.0000 | Val Acc: 0.7575
Accuracy: 0.7575 | Precision: 0.8111 | Recall: 0.7316 | F1: 0.7587
Best Params: {'colsample_bytree': np.float64(0.5488360570031919), 'gamma': np.float64(3.4211651325607844), 'learning_rate': np.float64(0.1420457481218804), 'max_depth': 9, 'n_estimators': 93, 'reg_alpha': np.float64(0.4951769101112702), 'reg_lambda': np.float64(0.034388521115218396), 'subsample': np.float64(0.954660201039391)}
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.47      0.64        15
           1       0.91      0.94      0.92       226
           2       0.85      0.58      0.69        19
           3       0.65      0.73      0.69       202
           4       0.73      0.67      0.70       153
           5       0.72      0.69      0.70        67
           6       0.60      0.60      0.60        75
           7       1.00      1.00   